In [22]:
import re
import json

In [110]:
class BPEMethod:
    def __init__(self, file_path: str):
        with open(file_path) as file:
            self.text = self.format_text(file.read())

    def format_text(self, text: str):
        text = text.lower()
        text = re.sub(r'[^а-яёa-z0-9 ,.!?-]', '', text)
        text = re.sub(r' +', ' ', text)

        return text

    def tokenize_text(self):
        t = re.findall(r'\[[0-9]+\]|.', self.text)
        # result = [num if num else char for num, char in t]
        return t

    def get_counts(self):
        counts = {}
        tokens = self.tokenize_text()
        for a, b in zip(tokens, tokens[1:]):
            key = (a, b)
            counts[key] = counts.get(key, 0) + 1
        return counts
            
    def check_bytes(self, codes: dict):
        format_codes = {k: f'{v[0]}{v[1]}' for k, v in codes.items()}
        json_string = json.dumps(format_codes, separators=[':', ','])
        return len(json_string.encode('utf-8')) <= 1024

    def replace_pair(self, index: int, pair: tuple[str]):
        str_pair = ''.join(pair)
        self.text = self.text.replace(str_pair, f'[{index}]')

    def bpe_compress(self):

        codes = {}

        while True:
            counts = self.get_counts()
            if not counts or max(counts.values()) == 1:
                break
            
            max_pair = max(counts, key=counts.get)
            last_key = max(codes) if codes else -1

            test_codes = codes.copy()
            test_codes[last_key + 1] = max_pair
            if not self.check_bytes(test_codes):
                break

            codes[last_key + 1] = max_pair
            self.replace_pair(index=last_key+1, pair=max_pair)
        
        return codes
        

In [111]:
test = BPEMethod('text.txt')
print(test.get_counts())

{('м', 'и'): 504, ('и', 'х'): 192, ('х', 'а'): 71, ('а', 'и'): 19, ('и', 'л'): 669, ('л', ' '): 811, (' ', 'б'): 692, ('б', 'у'): 115, ('у', 'л'): 257, ('л', 'г'): 14, ('г', 'а'): 199, ('а', 'к'): 479, ('к', 'о'): 1203, ('о', 'в'): 1337, ('в', '.'): 46, ('.', 'р'): 9, ('р', 'о'): 1115, ('о', 'к'): 341, ('в', 'ы'): 478, ('ы', 'е'): 207, ('е', ' '): 1557, (' ', 'я'): 201, ('я', 'й'): 46, ('й', 'ц'): 45, ('ц', 'а'): 119, ('а', 'г'): 126, ('г', 'л'): 234, ('л', 'а'): 736, ('а', 'в'): 315, ('в', 'а'): 784, ('а', ' '): 1581, (' ', '1'): 53, ('1', '.'): 2, ('.', 'к'): 22, ('к', 'у'): 299, ('у', 'р'): 174, ('р', 'р'): 12, ('р', 'и'): 602, ('и', 'к'): 506, ('л', 'ю'): 84, ('ю', 'м'): 4, ('м', ' '): 702, (' ', 'в'): 2020, ('в', 'и'): 458, ('и', 'т'): 407, ('т', 'э'): 2, ('э', ' '): 1, (' ', 'п'): 2336, ('п', 'р'): 849, ('о', 'ф'): 173, ('ф', 'е'): 172, ('е', 'с'): 650, ('с', 'с'): 231, ('с', 'о'): 486, ('о', 'р'): 780, ('р', 'а'): 1054, ('п', 'е'): 520, ('е', 'р'): 1136, ('р', 'с'): 283, ('с', '

In [112]:
print(test.bpe_compress())

{0: (',', ' '), 1: (' ', 'п'), 2: ('о', ' '), 3: ('и', ' '), 4: ('а', ' '), 5: ('е', ' '), 6: ('о', 'в'), 7: ('с', 'т'), 8: ('а', 'л'), 9: ('е', 'р'), 10: ('р', 'о'), 11: ('р', 'а'), 12: (' ', 'в'), 13: ('е', 'н'), 14: (' ', 'н'), 15: ('к', 'о'), 16: ('е', 'л'), 17: ('к', 'а'), 18: ('т', 'о'), 19: (' ', 'с'), 20: ('н', 'о'), 21: ('[1]', 'о'), 22: ('и', 'л'), 23: ('е', 'т'), 24: ('а', 'н'), 25: ('й', ' '), 26: ('я', ' '), 27: ('н', 'ы'), 28: ('.', ' '), 29: ('е', 'с'), 30: ('з', 'а'), 31: ('н', 'а'), 32: ('е', 'м'), 33: ('о', 'т'), 34: ('о', 'л'), 35: ('р', 'и'), 36: ('т', '[2]'), 37: ('ь', ' '), 38: ('и', 'к'), 39: ('и', 'н'), 40: ('о', 'н'), 41: ('г', '[2]'), 42: ('л', 'а'), 43: ('и', 'т'), 44: ('м', ' '), 45: ('о', 'р'), 46: ('о', 'д'), 47: ('у', ' '), 48: ('б', 'ы'), 49: ('[0]', 'п'), 50: ('в', ' '), 51: ('н', '[2]'), 52: ('и', 'з'), 53: ('а', 'т'), 54: ('в', 'ы'), 55: ('р', 'у')}


In [96]:
t = '[1112]aaa[12]dd[1]dd ,. ddd'
print(re.findall(r'\[([0-9]+)\]|(.)', t))

[('1112', ''), ('', 'a'), ('', 'a'), ('', 'a'), ('12', ''), ('', 'd'), ('', 'd'), ('1', ''), ('', 'd'), ('', 'd'), ('', ' '), ('', ','), ('', '.'), ('', ' '), ('', 'd'), ('', 'd'), ('', 'd')]


In [113]:
print(test.text)
print(test.format_text(test.text))

миха[22] булгак[6].[10]к[6]ы[5]яйцаг[42]в[4]1.кур[35]кулюм[12][43]э[1][10]ф[29]со[11][1][9]с[38][6]а16 апр[16][26]1928 г[46]а[0]веч[9]ом[49][10]ф[29]с[45] зо[34]оги[3]iv государ[7]в[13][20][41]унив[9]с[43][23][4][3]дирек[18]р зоо[39][7][43]ут[4][50]москве[1][9]с[38][6][12]ош[16][12][19]во[25][17]б[39][23][49]омещающийся[12] зоо[39][7][43]уте[0]ч[36]н[4]улиц[5]г[9]ц[13]а.[1][10]ф[29]с[45] [30]жег[12][9]хни[25]м[53][6]ы[25]шар [3]огляд[16]ся.[31]ч[8][2]ужасающе[25][17]та[7][10]фы[14]уж[51]сч[43][53][37]з[8]ож[13][27][44]им[13][51][50]э[18]т злосча[7][27]й[12]еч[9][0][11]в[51][17]к[1][9]воп[35]чи[20]ю э[18][25][17]та[7][10]фы[19]леду[23][19]ч[43][53][37]им[13][20][1][10]ф[29]с[45][4]в[42]димир[4]ип[53]ьевича[1][9]с[38][6]а.[32][47][48]л[2]р[6][51]58 л[23][28]г[34][6][4][30]меч[53][16]ь[31]я[0][18]л[17]чом[0]лысая[0]с[1]уч[17]м[3]ж[16]т[6][53]ых[12][34]ос[0][18]рчащими[1][2]бо[17]м[28]лиц[2]г[42]дк[2][54]б[35][18]е[0]нижня[26]губ[4][54]пяч[13][4]вп[9]ед[28][33] э[18]го[1][9]с[38][6]с[15][5